In [ ]:
import os
import pickle
import uuid
from tqdm.auto import tqdm

from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

In [ ]:
SAVE_DIR = "data/processed"
SPLIT_DOCS_PATH = os.path.join(SAVE_DIR, "split_docs.pkl")

CHROMA_DIR = "data/vectorstore/chroma_db"
COLLECTION_NAME = "my_documents"

OLLAMA_BASE_URL = "http://localhost:11434"
EMBED_MODEL = "bge-m3"

BATCH_SIZE = 32

In [ ]:
# Load split_docs.pkl
with open(SPLIT_DOCS_PATH, "rb") as f:
    split_docs = pickle.load(f)

print(f"Loaded {len(split_docs)} chunks")
print(type(split_docs[0]))
print(split_docs[0].metadata)

In [ ]:
# Create embedding model
embedding = OllamaEmbeddings(
    model=EMBED_MODEL,
    base_url=OLLAMA_BASE_URL
)

# sanity check
test_vec = embedding.embed_query("hello world")
print("Embedding dimension:", len(test_vec))

In [ ]:
# Create / save Chroma vector DB

os.makedirs(CHROMA_DIR, exist_ok=True)

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding,
    persist_directory=CHROMA_DIR
)

print("Chroma DB initialized")

In [ ]:
# Add documents in batches with tqdm
for start in tqdm(range(0, len(split_docs), BATCH_SIZE), desc="Embedding + storing"):
    end = min(start + BATCH_SIZE, len(split_docs))
    batch_docs = split_docs[start:end]
    batch_ids = [str(uuid.uuid4()) for _ in batch_docs]

    vector_store.add_documents(
        documents=batch_docs,
        ids=batch_ids
    )

print("All documents embedded and stored in Chroma successfully")
print(f"Saved at: {CHROMA_DIR}")

In [ ]:
# Test retrieval immediately

retriever = vector_store.as_retriever(search_kwargs={"k": 5})

query = "What is the main topic discussed in these documents?"
results = retriever.invoke(query)

print(f"Top {len(results)} results for query: {query}\n")

for i, doc in enumerate(results, 1):
    print(f"--- Result {i} ---")
    print(doc.page_content[:800])
    print("Metadata:", doc.metadata)
    print()

In [ ]:
# =========================
# 8) ADD DOCUMENTS IN BATCHES WITH tqdm
# =========================
total_docs = len(split_docs)

for start in tqdm(range(0, total_docs, BATCH_SIZE), desc="Embedding + Uploading"):
    end = min(start + BATCH_SIZE, total_docs)
    batch_docs = split_docs[start:end]

    # stable unique ids
    batch_ids = [str(uuid.uuid4()) for _ in batch_docs]

    vector_store.add_documents(documents=batch_docs, ids=batch_ids)

print(f"\nDone. Embedded and stored {total_docs} chunks in Qdrant collection: {COLLECTION_NAME}")

In [ ]:
# =========================
# 9) TEST RETRIEVAL
# =========================
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

query = "What is the main topic discussed in these documents?"
results = retriever.invoke(query)

print(f"Top {len(results)} results for query: {query}\n")

for i, doc in enumerate(results, 1):
    print(f"--- Result {i} ---")
    print(doc.page_content[:800])
    print("Metadata:", doc.metadata)
    print()

In [ ]:
# Reload saved DB
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

CHROMA_DIR = "data/vectorstore/chroma_db"
COLLECTION_NAME = "my_documents"

embedding = OllamaEmbeddings(
    model="bge-m3",
    base_url="http://localhost:11434"
)

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding,
    persist_directory=CHROMA_DIR
)

print("Existing Chroma DB loaded successfully")

In [ ]:
# Retrieve from loaded DB
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

query = "Give me summary of flood susceptibility information"
results = retriever.invoke(query)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:800])
    print(doc.metadata)

In [ ]:
import os
import pickle
import uuid
from tqdm.auto import tqdm

from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

# =========================
# CONFIG
# =========================
SAVE_DIR = "data/processed"
SPLIT_DOCS_PATH = os.path.join(SAVE_DIR, "split_docs.pkl")

CHROMA_DIR = "data/vectorstore/chroma_db"
COLLECTION_NAME = "my_documents"

OLLAMA_BASE_URL = "http://localhost:11434"
EMBED_MODEL = "bge-m3"
BATCH_SIZE = 32

# =========================
# LOAD DOCS
# =========================
with open(SPLIT_DOCS_PATH, "rb") as f:
    split_docs = pickle.load(f)

print(f"Loaded {len(split_docs)} chunks")

# =========================
# EMBEDDING MODEL
# =========================
embedding = OllamaEmbeddings(
    model=EMBED_MODEL,
    base_url=OLLAMA_BASE_URL
)

test_vec = embedding.embed_query("hello world")
print("Embedding dimension:", len(test_vec))

# =========================
# CHROMA DB INIT
# =========================
os.makedirs(CHROMA_DIR, exist_ok=True)

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding,
    persist_directory=CHROMA_DIR
)

print("Chroma initialized")

# =========================
# ADD DOCS IN BATCHES
# =========================
for start in tqdm(range(0, len(split_docs), BATCH_SIZE), desc="Embedding + storing"):
    end = min(start + BATCH_SIZE, len(split_docs))
    batch_docs = split_docs[start:end]
    batch_ids = [str(uuid.uuid4()) for _ in batch_docs]

    vector_store.add_documents(
        documents=batch_docs,
        ids=batch_ids
    )

print("Vector DB created and saved successfully")

# =========================
# TEST RETRIEVAL
# =========================
retriever = vector_store.as_retriever(search_kwargs={"k": 5})
results = retriever.invoke("What is the main topic discussed in these documents?")

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:800])
    print(doc.metadata)